# REE Lambda Fitting — Part 3: Plotting

Loads fitted results and produces three figure types:

1. **Pairwise χ² trade-off matrix** — for one sample, shows how each pair of
   λᵢ / λⱼ trade off against each other (Δχ² contours)
2. **Population scatter grid** — all samples plotted as λᵢ vs λⱼ,
   coloured by temperature or depth
3. **Pyrolite comparison** — validates custom λ values against pyrolite's
   implementation on a 1:1 plot for each λ

**Inputs**
- `df_analysis.parquet` — from `01_data_prep.ipynb`
- `df_model.parquet` — from `02_lambda_fitting.ipynb`


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pyrolite.plot


## 2. Constants

In [ ]:
REE = ["La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"]

IONIC_RADII = np.array([
    1.160, 1.143, 1.126, 1.109, 1.079, 1.066,
    1.053, 1.040, 1.027, 1.015, 1.004, 0.994,
    0.985, 0.977
], dtype=float)

CI = np.array([
    0.2472, 0.6308, 0.0950, 0.4793, 0.1542, 0.0592,
    0.2059, 0.0375, 0.2540, 0.0554, 0.1645, 0.0258,
    0.1684, 0.0251
], dtype=float)

LAM_COLS = ["λ0", "λ1", "λ2", "λ3", "λ4"]


## 3. Load data

In [ ]:
df_analysis = pd.read_csv("df_analysis.csv", index_col=0)
df_model    = pd.read_csv("df_model.csv",    index_col=0)

print("df_analysis:", df_analysis.shape)
print("df_model   :", df_model.shape)


## 4. Helper functions (shared with fitting notebook)

In [ ]:
def oneill_2016_basis(r, degree=4):
    r  = np.asarray(r, dtype=float)
    f0 = np.ones_like(r)
    f1 = (r - 1.05477)
    f2 = (r - 1.00533) * (r - 1.12824)
    f3 = (r - 0.99141) * (r - 1.06055) * (r - 1.14552)
    f4 = (r - 0.98482) * (r - 1.03052) * (r - 1.10441) * (r - 1.15343)
    cols = [f0, f1, f2, f3, f4]
    return np.column_stack(cols[: degree + 1])


def _safe_ln_ratio(x, ci):
    x, ci = np.asarray(x, float), np.asarray(ci, float)
    y     = np.full_like(x, np.nan, dtype=float)
    good  = np.isfinite(x) & np.isfinite(ci) & (x > 0) & (ci > 0)
    y[good] = np.log(x[good] / ci[good])
    return y, good


def fit_lambdas_ols(F, y):
    beta, *_ = np.linalg.lstsq(F, y, rcond=1e-12)
    return beta


def _ols_cov_se(F, y, beta):
    n, p  = F.shape
    resid = y - F @ beta
    dof   = n - p
    if dof <= 0:
        return None, None, dof, np.nan, resid
    sse     = float(np.sum(resid**2))
    s2      = sse / dof
    XtX_inv = np.linalg.pinv(F.T @ F)
    cov     = s2 * XtX_inv
    se      = np.sqrt(np.diag(cov))
    return cov, se, dof, s2, resid


## 5. Plot 1 — Pairwise χ² trade-off matrix (single sample)

Shows, for one sample, how each pair of λ values trade off.
Lower-triangle panels show Δχ² contours; the OLS best-fit is marked with `+`.
Circular contours = independent parameters; elongated/tilted = correlated.


In [ ]:
def pairwise_tradeoffs_all_lambdas_pubstyle(
    df_model,
    row_id,
    REE,
    radii,
    ci,
    degree=4,
    exclude=(),
    sigma_ln=0.02,
    grid_n=100,
    half_width_sigma=3.0,
    contour_levels=(1, 3, 6, 12),
    filled_levels=120,
    figsize_scale=3.0,
    title=None,
    show_diagonal_names=True,
    show_upper_triangle=False,
):
    """
    Publication-style full pairwise lambda trade-off matrix.

    Lower triangle : Δχ² contour panels
    Diagonal       : lambda names
    Upper triangle : blank by default
    """
    # Build y = ln(REE/CI)
    sample_ppm = df_model.loc[row_id, REE].to_numpy(float)
    y_all, good_all = _safe_ln_ratio(sample_ppm, ci)

    excl_mask = np.ones(len(REE), dtype=bool)
    for el in exclude:
        if el in REE:
            excl_mask[REE.index(el)] = False

    fit_mask      = good_all & excl_mask
    x             = np.asarray(radii, float)[fit_mask]
    y             = np.asarray(y_all,  float)[fit_mask]
    used_elements = [e for e, m in zip(REE, fit_mask) if m]

    p = degree + 1
    if y.size < p:
        raise ValueError(f"Not enough usable REEs for degree={degree}: need {p}, have {y.size}.")

    # OLS centre
    F       = oneill_2016_basis(x, degree=degree)
    lam_ols = np.asarray(fit_lambdas_ols(F, y), float)[:p]

    se = np.ones(p, float)
    try:
        cov, se_est, dof, s2, resid = _ols_cov_se(F, y, lam_ols)
        if se_est is not None:
            se = np.asarray(se_est, float)[:p]
    except Exception:
        pass
    se[~np.isfinite(se)] = 1.0
    se[se == 0]          = 1.0

    # χ² surface for one (i, j) pair
    def chi2_surface_for_pair(i, j, grid_x, grid_y):
        cols  = np.arange(p)
        fixed = np.array([i, j], dtype=int)
        free  = np.array([k for k in cols if k not in fixed], dtype=int)

        Fi    = F[:, i]
        Fj    = F[:, j]
        Ffree = F[:, free]

        chi2 = np.empty((len(grid_y), len(grid_x)), float)
        for a, lam_i in enumerate(grid_y):
            for b, lam_j in enumerate(grid_x):
                y_res      = y - (lam_i * Fi) - (lam_j * Fj)
                lam_free, *_ = np.linalg.lstsq(Ffree, y_res, rcond=1e-12)
                y_pred     = (lam_i * Fi) + (lam_j * Fj) + (Ffree @ lam_free)
                resid      = y - y_pred
                chi2[a, b] = np.sum((resid / sigma_ln)**2)
        return chi2

    # Build figure
    lam_names = [f"λ{k}" for k in range(p)]
    fig, axes = plt.subplots(
        p, p,
        figsize=(figsize_scale * p, figsize_scale * p),
        constrained_layout=True
    )
    if title is None:
        title = f"Pairwise λ trade-offs (Δχ²), row_id={row_id}"
    fig.suptitle(title, y=1.01, fontsize=13)

    for i in range(p):
        for j in range(p):
            ax = axes[i, j]
            if i < j and not show_upper_triangle:
                ax.axis("off"); continue
            if i == j:
                if show_diagonal_names:
                    ax.text(0.5, 0.5, lam_names[i], ha="center", va="center",
                            transform=ax.transAxes, fontsize=14)
                ax.set_xticks([]); ax.set_yticks([])
                for sp in ax.spines.values(): sp.set_linewidth(1.0)
                continue

            x0, y0 = lam_ols[j], lam_ols[i]
            wx, wy  = half_width_sigma * se[j], half_width_sigma * se[i]
            grid_x  = np.linspace(x0 - wx, x0 + wx, grid_n)
            grid_y  = np.linspace(y0 - wy, y0 + wy, grid_n)

            chi2  = chi2_surface_for_pair(i, j, grid_x, grid_y)
            dchi2 = chi2 - np.nanmin(chi2)
            X, Y  = np.meshgrid(grid_x, grid_y)

            ax.contourf(X, Y, dchi2, levels=filled_levels, cmap="Greys")
            cs = ax.contour(X, Y, dchi2, levels=contour_levels,
                            colors="blue", linewidths=0.8)
            ax.clabel(cs, inline=True, fontsize=7, fmt="%g")
            ax.plot(x0, y0, marker="+", ms=9, mew=1.4, color="k")

            if i == p - 1: ax.set_xlabel(lam_names[j])
            else:           ax.set_xticklabels([])
            if j == 0:      ax.set_ylabel(lam_names[i])
            else:            ax.set_yticklabels([])

            ax.tick_params(direction="in", top=True, right=True)
            for sp in ax.spines.values(): sp.set_linewidth(1.0)

    plt.show()
    return {"row_id": row_id, "degree": degree, "exclude": exclude,
            "sigma_ln": sigma_ln, "used_elements": used_elements,
            "x_used": x, "y_used": y, "lambdas_ols": lam_ols, "se_ols": se}


In [ ]:
# ── Run for one sample ────────────────────────────────────────────────────────
out_tradeoffs = pairwise_tradeoffs_all_lambdas_pubstyle(
    df_model        = df_model,
    row_id          = 169,
    REE             = REE,
    radii           = IONIC_RADII,
    ci              = CI,
    degree          = 4,
    sigma_ln        = 0.1,
    grid_n          = 100,
    half_width_sigma= 3.0,
    contour_levels  = (1, 4, 9),
    filled_levels   = 25,
    figsize_scale   = 3.0,
    title           = "λ trade-offs",
)


## 6. Plot 2 — Population λᵢ vs λⱼ scatter grid

All samples shown as pairwise scatter plots, coloured by temperature or depth.


In [ ]:
def plot_lambda_population_grid(
    df_model,
    color_by="Temp (Celc)",
    degree=4,
    figsize_scale=2.8,
    alpha=0.65,
    s=18,
    cmap="plasma",
):
    """
    Pairwise scatter grid of λ0–λ4 for the full population.
    Lower triangle: λᵢ vs λⱼ coloured by `color_by`.
    Diagonal: λ name labels. Upper triangle: blank.
    """
    p        = degree + 1
    lam_cols = [f"λ{k}" for k in range(p)]

    missing = [c for c in lam_cols if c not in df_model.columns]
    if missing:
        raise ValueError(f"Missing lambda columns: {missing}")
    if color_by not in df_model.columns:
        raise ValueError(f"Column '{color_by}' not found.")

    mask     = np.isfinite(df_model[lam_cols].values).all(axis=1)
    mask    &= np.isfinite(df_model[color_by].values)
    df_plot  = df_model.loc[mask].copy()
    if len(df_plot) == 0:
        raise ValueError("No rows remain after filtering.")

    print(f"Plotting {len(df_plot)} samples (of {len(df_model)}) coloured by '{color_by}'")

    c_vals   = df_plot[color_by].values
    norm     = mcolors.Normalize(vmin=np.nanpercentile(c_vals, 2),
                                  vmax=np.nanpercentile(c_vals, 98))
    cmap_obj = matplotlib.colormaps[cmap]
    colours  = cmap_obj(norm(c_vals))

    fig, axes = plt.subplots(p, p,
                             figsize=(figsize_scale * p, figsize_scale * p),
                             constrained_layout=True)
    color_label = color_by.replace(" (Celc)", " (°C)")
    fig.suptitle(
        f"Population λ trade-offs — {len(df_plot)} samples, coloured by {color_label}",
        fontsize=13, y=1.01
    )

    def iqr_limits(arr, k=2.5):
        q1, q3 = np.nanpercentile(arr, 25), np.nanpercentile(arr, 75)
        iqr = q3 - q1
        return q1 - k * iqr, q3 + k * iqr

    for i in range(p):
        for j in range(p):
            ax = axes[i, j]
            if i < j:
                ax.axis("off"); continue
            if i == j:
                ax.text(0.5, 0.5, lam_cols[i], ha="center", va="center",
                        transform=ax.transAxes, fontsize=15, fontweight="bold")
                ax.set_xticks([]); ax.set_yticks([])
                for sp in ax.spines.values(): sp.set_linewidth(0.8)
                continue

            x_data = df_plot[lam_cols[j]].values
            y_data = df_plot[lam_cols[i]].values
            ax.scatter(x_data, y_data, c=colours, s=s, alpha=alpha,
                       linewidths=0, rasterized=True)
            ax.set_xlim(*iqr_limits(x_data))
            ax.set_ylim(*iqr_limits(y_data))
            ax.axhline(0, color="0.75", linewidth=0.6, zorder=0)
            ax.axvline(0, color="0.75", linewidth=0.6, zorder=0)

            if i == p - 1: ax.set_xlabel(lam_cols[j], fontsize=10)
            else:           ax.set_xticklabels([])
            if j == 0:      ax.set_ylabel(lam_cols[i], fontsize=10)
            else:            ax.set_yticklabels([])

            ax.tick_params(direction="in", top=True, right=True, labelsize=7)
            for sp in ax.spines.values(): sp.set_linewidth(0.8)

    sm   = cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, orientation="vertical",
                        fraction=0.015, pad=0.02, aspect=35)
    cbar.set_label(color_label, fontsize=11)
    cbar.ax.tick_params(labelsize=9)

    plt.show()
    return fig


In [ ]:
import matplotlib

fig_temp = plot_lambda_population_grid(
    df_model,
    color_by="Temp (Celc)",
    cmap="plasma",
    alpha=0.65,
    s=20,
)

# Uncomment to colour by depth instead:
# fig_depth = plot_lambda_population_grid(
#     df_model,
#     color_by="Depth (km)",
#     cmap="viridis_r",
#     alpha=0.65,
#     s=20,
# )


## 7. Plot 3 — Pyrolite vs custom λ comparison

Validates the custom OLS implementation against pyrolite's `lambda_lnREE`.
Each panel shows one λ on a 1:1 plot with a residual histogram inset.


In [ ]:
def centred_limits(x, y, lo=0.02, hi=0.98, centre=0.0):
    """Robust symmetric axis limits centred on `centre`, IQR-clipped."""
    v = np.concatenate([x, y])
    v = v[np.isfinite(v)]
    if v.size == 0:
        return (-1, 1)
    q1, q3 = np.percentile(v, 25), np.percentile(v, 75)
    iqr    = q3 - q1
    v      = v[(v >= q1 - 2.5 * iqr) & (v <= q3 + 2.5 * iqr)]
    if v.size == 0:
        return (-1, 1)
    q_lo, q_hi = np.quantile(v, [lo, hi])
    span = max(abs(q_lo - centre), abs(q_hi - centre), 1e-9)
    return (centre - span, centre + span)


def plot_lambda_comparison(
    comparison,
    color_by="Temp (Celc)",
    cmap="plasma",
    alpha=0.65,
    s=20,
    figsize_per_panel=(5, 5.8),
):
    """One 1:1 panel per λ comparing pyrolite vs custom values."""
    plot_lams = [
        c for c in LAM_COLS
        if f"{c}_pyro" in comparison.columns and f"{c}_custom" in comparison.columns
    ]
    n = len(plot_lams)
    if n == 0:
        raise ValueError("No λ_pyro / λ_custom column pairs found.")

    use_colour  = color_by is not None and color_by in comparison.columns
    color_label = (color_by or "").replace(" (Celc)", " (°C)")

    if use_colour:
        c_vals   = comparison[color_by].to_numpy(float)
        finite_c = c_vals[np.isfinite(c_vals)]
        norm     = mcolors.Normalize(vmin=np.percentile(finite_c, 2),
                                      vmax=np.percentile(finite_c, 98))
        cmap_obj = matplotlib.colormaps[cmap]

    fw, fh    = figsize_per_panel
    fig, axes = plt.subplots(1, n, figsize=(fw * n, fh), constrained_layout=True)
    if n == 1:
        axes = [axes]

    fig.suptitle(
        "Pyrolite vs Custom λ" + (f"  |  coloured by {color_label}" if use_colour else ""),
        fontsize=14, y=1.02,
    )

    for ax, lam in zip(axes, plot_lams):
        x = comparison[f"{lam}_pyro"].to_numpy(float)
        y = comparison[f"{lam}_custom"].to_numpy(float)
        m = np.isfinite(x) & np.isfinite(y)
        if use_colour: m &= np.isfinite(c_vals)
        xm, ym = x[m], y[m]

        colours = cmap_obj(norm(c_vals[m])) if use_colour else "steelblue"
        ax.scatter(xm, ym, c=colours, s=s, alpha=alpha, linewidths=0, rasterized=True)

        lo, hi = centred_limits(xm, ym)
        ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.0, zorder=3)
        ax.axhline(0, color="0.80", linewidth=0.6, zorder=0)
        ax.axvline(0, color="0.80", linewidth=0.6, zorder=0)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_aspect("equal", adjustable="box")

        resid = ym - xm
        rmse  = np.sqrt(np.mean(resid**2))
        bias  = np.mean(resid)
        r     = np.corrcoef(xm, ym)[0, 1] if xm.size > 1 else np.nan
        ax.text(
            0.04, 0.97,
            f"n = {xm.size}\nr = {r:.3f}\nRMSE = {rmse:.4f}\nbias = {bias:+.4f}",
            transform=ax.transAxes, fontsize=8, va="top", ha="left",
            family="monospace",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.75", alpha=0.85),
        )

        ins = ax.inset_axes([0.62, 0.04, 0.35, 0.28])
        ins.hist(resid, bins=20, color="0.55", edgecolor="white", linewidth=0.4)
        ins.axvline(0,    color="k",       linewidth=0.8)
        ins.axvline(bias, color="crimson", linewidth=0.8, linestyle="--")
        ins.set_xlabel("custom − pyro", fontsize=6)
        ins.set_ylabel("count",          fontsize=6)
        ins.tick_params(labelsize=5)
        for sp in ins.spines.values(): sp.set_linewidth(0.5)

        ax.set_xlabel(f"{lam}  (pyrolite)", fontsize=11)
        ax.set_ylabel(f"{lam}  (custom)",   fontsize=11)
        ax.set_title(lam, fontsize=13, fontweight="bold")
        ax.tick_params(direction="in", top=True, right=True, labelsize=8)

    if use_colour:
        sm = cm.ScalarMappable(cmap=cmap_obj, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=axes, fraction=0.012, pad=0.02, aspect=35, shrink=0.7)
        cbar.set_label(color_label, fontsize=11)
        cbar.ax.tick_params(labelsize=9)

    plt.show()
    return fig


In [ ]:
# ── Build pyrolite lambda table ───────────────────────────────────────────────
lp = (
    df_analysis[REE]
    .pyrochem.lambda_lnREE(algorithm="ONeill", degree=5, exclude=[])
    .copy()
)
lp.index = df_analysis.index

lp.columns = [str(c) for c in lp.columns]
rename_map = {
    "lambda0": "λ0", "lambda1": "λ1", "lambda2": "λ2", "lambda3": "λ3", "lambda4": "λ4",
    "0": "λ0",       "1": "λ1",       "2": "λ2",       "3": "λ3",       "4": "λ4",
}
lp = lp.rename(columns={k: v for k, v in rename_map.items() if k in lp.columns})

pyro_lams = [c for c in LAM_COLS if c in lp.columns]
print("Pyrolite lambdas returned:", pyro_lams)
if "λ4" not in pyro_lams:
    print("WARNING: λ4 missing — check that pyrolite ran with degree=4.")

# ── Join and compute differences ─────────────────────────────────────────────
sample_col = "SAMPLE NO."
custom_df  = df_model[[sample_col] + LAM_COLS].copy()
custom_df  = custom_df.rename(columns={c: f"{c}_custom" for c in LAM_COLS})

lp_renamed = lp[pyro_lams].rename(columns={c: f"{c}_pyro" for c in pyro_lams})
comparison = custom_df.join(lp_renamed, how="inner")

for c in pyro_lams:
    comparison[f"Δ{c}"] = comparison[f"{c}_custom"] - comparison[f"{c}_pyro"]

for col in ["Temp (Celc)", "Depth (km)"]:
    if col in df_model.columns:
        comparison[col] = df_model[col]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_comp = plot_lambda_comparison(comparison, color_by="Temp (Celc)", cmap="plasma")
